# DeBERTa-v3 Student Misconception Classification with Explainable AI

**Kaggle-ready training script for classifying student misconceptions**

Optimized for Kaggle GPU (Tesla T4/P100)

## Features:
- DeBERTa-v3 transformer model
- Multi-class classification
- **Explainable AI (XAI)** - Model explanations with:
  - Attention visualization
  - Token importance analysis
  - Human-readable reasoning
  - SHAP & LIME support

## To run on Kaggle:
1. Create a new notebook
2. Add your dataset (train.csv, test.csv)
3. Enable GPU accelerator (Settings > Accelerator > GPU T4 x2 or P100)
4. Run all cells

## Section 1: Install & Import Dependencies

In [ ]:
# Install required packages (uncomment for first-time setup)
# !pip install -q transformers datasets accelerate sentencepiece shap lime

In [ ]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.special import softmax
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import pickle
import matplotlib.pyplot as plt
from collections import defaultdict

# Explainable AI imports
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠ SHAP not installed. Run: pip install shap")

try:
    from lime.lime_text import LimeTextExplainer
    LIME_AVAILABLE = True
except ImportError:
    LIME_AVAILABLE = False
    print("⚠ LIME not installed. Run: pip install lime")

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup
)
from datasets import Dataset as HFDataset

warnings.filterwarnings("ignore")

print("✓ All packages imported successfully!")

## Section 2: Configuration

In [ ]:
class Config:
    """Central configuration for all hyperparameters"""
    
    # Paths - Update these for Kaggle
    TRAIN_PATH = "/kaggle/input/dataset2/train.csv"  # Update with your dataset path
    OUTPUT_DIR = "/kaggle/working/deberta_output"
    
    # For local testing, use these paths:
    #TRAIN_PATH = "train.csv"
    #OUTPUT_DIR = "./deberta_output"
    
    # Model
    MODEL_NAME = "microsoft/deberta-v3-base"  # Options: deberta-v3-base, deberta-v3-large
    MAX_LENGTH = 256
    
    # Training
    EPOCHS = 3
    BATCH_SIZE = 8          # Reduce to 4 if OOM
    EVAL_BATCH_SIZE = 16
    GRADIENT_ACCUMULATION = 2  # Effective batch = 16
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    
    # Regularization
    DROPOUT = 0.1
    LABEL_SMOOTHING = 0.1
    MAX_GRAD_NORM = 1.0
    
    # Cross-validation
    N_FOLDS = 5
    USE_CV = False  # Set to True for full CV training
    
    # Misc
    SEED = 42
    FP16 = True
    NUM_WORKERS = 2
    MIN_CLASS_SAMPLES = 5  # Minimum samples per class
    
    # Debug mode - set True for quick testing
    DEBUG = False          # DISABLED for full training
    DEBUG_SAMPLES = 1000

In [ ]:
def seed_everything(seed):
    """Set seeds for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def print_gpu_info():
    """Print GPU information"""
    print("=" * 60)
    print("GPU INFORMATION")
    print("=" * 60)
    
    if torch.cuda.is_available():
        print(f"✓ CUDA Available: True")
        print(f"✓ GPU Device: {torch.cuda.get_device_name(0)}")
        
        try:
            free, total = torch.cuda.mem_get_info()
            print(f"✓ GPU Memory: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")
        except:
            total_mem = torch.cuda.get_device_properties(0).total_memory
            print(f"✓ GPU Memory: {total_mem/1e9:.2f} GB total")
    else:
        print("⚠ No GPU available - training will be slow!")
    
    print("=" * 60)

## Section 3: Data Loading & Preprocessing

In [ ]:
def load_and_preprocess_data(config):
    """Load and preprocess the training data"""
    
    print("\n[STEP 1/6] Loading Data...")
    print("-" * 40)
    
    # Load data
    train_df = pd.read_csv(config.TRAIN_PATH)
    
    print(f"Original dataset size: {len(train_df)}")
    
    # Fill missing values
    train_df["StudentExplanation"] = train_df["StudentExplanation"].fillna("")
    train_df["QuestionText"] = train_df["QuestionText"].fillna("")
    train_df["MC_Answer"] = train_df["MC_Answer"].fillna("")
    train_df["Misconception"] = train_df["Misconception"].fillna("NA")
    train_df["Category"] = train_df["Category"].fillna("Unknown")
    
    # Create combined label
    train_df["label"] = train_df["Category"] + " | " + train_df["Misconception"]
    
    # Remove rows with missing labels
    train_df = train_df.dropna(subset=["label", "StudentExplanation"])
    
    # Filter rare classes
    class_counts = train_df["label"].value_counts()
    rare_classes = class_counts[class_counts < config.MIN_CLASS_SAMPLES].index
    train_df = train_df[~train_df["label"].isin(rare_classes)]
    
    print(f"After filtering rare classes: {len(train_df)}")
    print(f"Number of classes: {train_df['label'].nunique()}")
    
    # Debug mode - use subset
    if config.DEBUG:
        print("\n⚡ DEBUG MODE ENABLED - Using small subset for quick testing")
        # Take top N most common classes to ensure enough samples
        top_classes = class_counts.head(20).index.tolist()
        train_df = train_df[train_df['label'].isin(top_classes)]
        # Sample from each class
        train_df = train_df.groupby('label', group_keys=False).apply(
            lambda x: x.sample(min(len(x), 10), random_state=config.SEED)
        ).reset_index(drop=True)
        print(f"DEBUG MODE: Using {len(train_df)} samples across {train_df['label'].nunique()} classes")
    
    # Encode labels
    label_encoder = LabelEncoder()
    train_df["label_id"] = label_encoder.fit_transform(train_df["label"])
    num_labels = len(label_encoder.classes_)
    
    print(f"✓ Final samples: {len(train_df)}")
    print(f"✓ Number of classes: {num_labels}")
    
    return train_df, label_encoder, num_labels


def create_input_text(row):
    """Create formatted input text from dataframe row"""
    text = f"Question: {row['QuestionText']}\n"
    text += f"Student Answer: {row['MC_Answer']}\n"
    text += f"Student Explanation: {row['StudentExplanation']}"
    return text

In [ ]:
def prepare_datasets(train_df, tokenizer, config):
    """Prepare HuggingFace datasets for training"""
    
    print("\n[STEP 2/6] Preparing Datasets...")
    print("-" * 40)
    
    # Create text column
    train_df = train_df.copy()
    train_df["text"] = train_df.apply(create_input_text, axis=1)
    
    # Split data
    X_train, X_val = train_test_split(
        train_df,
        test_size=0.15,
        random_state=config.SEED,
        stratify=train_df["label_id"]
    )
    
    print(f"✓ Training samples: {len(X_train)}")
    print(f"✓ Validation samples: {len(X_val)}")
    
    # Create HuggingFace datasets
    train_dataset = HFDataset.from_pandas(
        X_train[["text", "label_id"]].rename(columns={"label_id": "labels"}).reset_index(drop=True)
    )
    
    val_dataset = HFDataset.from_pandas(
        X_val[["text", "label_id"]].rename(columns={"label_id": "labels"}).reset_index(drop=True)
    )
    
    # Tokenization function
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=config.MAX_LENGTH,
            padding=False  # Dynamic padding handled by DataCollator
        )
    
    # Tokenize datasets
    print("Tokenizing datasets...")
    train_dataset = train_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"],
        desc="Tokenizing train"
    )
    
    val_dataset = val_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"],
        desc="Tokenizing validation"
    )
    
    print("✓ Tokenization complete")
    
    return train_dataset, val_dataset, X_train, X_val

## Section 4: Model Setup

In [ ]:
def setup_model_and_tokenizer(config, num_labels):
    """Initialize model and tokenizer"""
    
    print("\n[STEP 3/6] Loading Model & Tokenizer...")
    print("-" * 40)
    print(f"Model: {config.MODEL_NAME}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    
    # Load model config
    model_config = AutoConfig.from_pretrained(
        config.MODEL_NAME,
        num_labels=num_labels,
        hidden_dropout_prob=config.DROPOUT,
        attention_probs_dropout_prob=config.DROPOUT,
    )
    
    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(
        config.MODEL_NAME,
        config=model_config,
    )
    
    # Enable gradient checkpointing for memory efficiency
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    model.config.use_cache = False
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"✓ Total parameters: {total_params:,}")
    print(f"✓ Trainable parameters: {trainable_params:,}")
    print("✓ Gradient checkpointing enabled")
    
    return model, tokenizer

## Section 5: Training

In [ ]:
def compute_metrics(eval_pred):
    """Compute evaluation metrics"""
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
    f1_weighted = f1_score(labels, preds, average='weighted', zero_division=0)
    
    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

In [ ]:
def train_model(model, tokenizer, train_dataset, val_dataset, config):
    """Train the model using HuggingFace Trainer"""
    
    print("\n[STEP 4/6] Training Model...")
    print("-" * 40)
    
    # Create output directory
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=config.OUTPUT_DIR,
        
        # Training settings
        num_train_epochs=config.EPOCHS,
        per_device_train_batch_size=config.BATCH_SIZE,
        per_device_eval_batch_size=config.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=config.GRADIENT_ACCUMULATION,
        
        # Optimizer settings
        learning_rate=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY,
        warmup_ratio=config.WARMUP_RATIO,
        max_grad_norm=config.MAX_GRAD_NORM,
        optim="adamw_torch",
        
        # Regularization
        label_smoothing_factor=config.LABEL_SMOOTHING,
        
        # Evaluation & Saving
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        
        # Precision & Performance
        fp16=config.FP16,
        dataloader_num_workers=config.NUM_WORKERS,
        
        # Logging
        logging_dir=f"{config.OUTPUT_DIR}/logs",
        logging_steps=50,
        report_to="none",
        
        # Misc
        seed=config.SEED,
        remove_unused_columns=True,
    )
    
    # Data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    # Callbacks
    callbacks = [
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
    
    # Initialize trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=callbacks
    )
    
    # Train
    print(f"Starting training for {config.EPOCHS} epochs...")
    print(f"Effective batch size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION}")
    
    train_result = trainer.train()
    
    print("\n✓ Training complete!")
    print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
    
    return trainer

## Section 6: Evaluation & Prediction

In [ ]:
def evaluate_model(trainer, val_dataset, X_val, label_encoder):
    """Evaluate model and compute MAP@3"""
    
    print("\n[STEP 5/6] Evaluating Model...")
    print("-" * 40)
    
    # Get predictions
    predictions = trainer.predict(val_dataset)
    probs = softmax(predictions.predictions, axis=1)
    
    # Top-3 predictions
    top3_idx = np.argsort(-probs, axis=1)[:, :3]
    top3_labels = [[label_encoder.classes_[i] for i in row] for row in top3_idx]
    
    # True labels
    true_labels = label_encoder.inverse_transform(X_val["label_id"].values)
    
    # Compute MAP@3
    def mean_average_precision_at_k(y_true, y_pred, k=3):
        """Calculate Mean Average Precision @ K"""
        scores = []
        for true, preds in zip(y_true, y_pred):
            for i, pred in enumerate(preds[:k]):
                if pred == true:
                    scores.append(1.0 / (i + 1))
                    break
            else:
                scores.append(0.0)
        return np.mean(scores)
    
    map_at_3 = mean_average_precision_at_k(true_labels, top3_labels, k=3)
    top1_accuracy = np.mean([preds[0] == true for preds, true in zip(top3_labels, true_labels)])
    top3_accuracy = np.mean([true in preds[:3] for preds, true in zip(top3_labels, true_labels)])
    
    print("\n" + "=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(f"MAP@3:          {map_at_3:.4f}")
    print(f"Top-1 Accuracy: {top1_accuracy:.4f}")
    print(f"Top-3 Accuracy: {top3_accuracy:.4f}")
    print("=" * 50)
    
    return {
        "map_at_3": map_at_3,
        "top1_accuracy": top1_accuracy,
        "top3_accuracy": top3_accuracy,
        "predictions": probs,
        "top3_labels": top3_labels
    }

In [ ]:
def save_validation_predictions(results, X_val, label_encoder, config):
    """Save validation predictions for analysis"""
    
    print("\n[STEP 6/6] Saving Validation Results...")
    print("-" * 40)
    
    # Create validation results DataFrame
    val_results = pd.DataFrame({
        "row_id": X_val["row_id"].values,
        "true_label": label_encoder.inverse_transform(X_val["label_id"].values),
        "prediction_1": [preds[0] for preds in results["top3_labels"]],
        "prediction_2": [preds[1] for preds in results["top3_labels"]],
        "prediction_3": [preds[2] for preds in results["top3_labels"]],
    })
    
    # Add correctness flags
    val_results["correct_top1"] = val_results["true_label"] == val_results["prediction_1"]
    val_results["correct_top3"] = val_results.apply(
        lambda x: x["true_label"] in [x["prediction_1"], x["prediction_2"], x["prediction_3"]], 
        axis=1
    )
    
    # Save
    results_path = f"{config.OUTPUT_DIR}/validation_predictions.csv"
    val_results.to_csv(results_path, index=False)
    print(f"✓ Validation predictions saved to: {results_path}")
    
    # Print some error analysis
    print("\n" + "=" * 50)
    print("ERROR ANALYSIS")
    print("=" * 50)
    
    incorrect = val_results[~val_results["correct_top1"]]
    print(f"Total incorrect (Top-1): {len(incorrect)}")
    
    if len(incorrect) > 0:
        print("\nSample misclassifications:")
        for _, row in incorrect.head(5).iterrows():
            print(f"  True: {row['true_label'][:50]}...")
            print(f"  Pred: {row['prediction_1'][:50]}...")
            print()
    
    return val_results

In [ ]:
def save_artifacts(trainer, label_encoder, config):
    """Save model and artifacts"""
    
    print("\nSaving model artifacts...")
    
    # Save model
    model_path = f"{config.OUTPUT_DIR}/final_model"
    trainer.save_model(model_path)
    print(f"✓ Model saved to: {model_path}")
    
    # Save label encoder
    encoder_path = f"{config.OUTPUT_DIR}/label_encoder.pkl"
    with open(encoder_path, 'wb') as f:
        pickle.dump(label_encoder, f)
    print(f"✓ Label encoder saved to: {encoder_path}")
    
    # Save config
    config_path = f"{config.OUTPUT_DIR}/config.pkl"
    with open(config_path, 'wb') as f:
        pickle.dump({
            'model_name': config.MODEL_NAME,
            'max_length': config.MAX_LENGTH,
            'num_labels': len(label_encoder.classes_)
        }, f)
    print(f"✓ Config saved to: {config_path}")

## Section 7: Explainable AI (XAI)

This section implements the **Explainable AI** module that provides human-readable explanations for the model's predictions.

### Features:
- **Attention-based explanations**: Visualize which parts of the input the model focused on
- **Token importance analysis**: Identify key words that influenced the prediction
- **Human-readable reasoning**: Generate natural language explanations
- **SHAP & LIME support**: Advanced feature importance methods

In [ ]:
class MisconceptionExplainer:
    """
    Explainable AI class for DeBERTa Student Misconception Classification.
    Provides multiple explanation methods:
    1. Attention-based explanations
    2. SHAP explanations
    3. LIME explanations
    4. Human-readable reasoning
    """
    
    def __init__(self, model, tokenizer, label_encoder, device=None):
        self.model = model
        self.tokenizer = tokenizer
        self.label_encoder = label_encoder
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.model.eval()
        
        # Store misconception descriptions for better explanations
        self.misconception_hints = self._build_misconception_hints()
    
    def _build_misconception_hints(self):
        """Build hint descriptions for common misconception types"""
        hints = {
            "calculation": "The student made an arithmetic or computational error",
            "conceptual": "The student has a fundamental misunderstanding of the concept",
            "procedural": "The student applied the wrong procedure or steps",
            "sign": "The student made a sign error (positive/negative confusion)",
            "unit": "The student made a unit conversion or dimensional error",
            "order": "The student performed operations in the wrong order",
            "interpretation": "The student misinterpreted the question or data",
            "formula": "The student used the wrong formula or equation",
            "boundary": "The student didn't consider edge cases or boundaries",
            "assumption": "The student made incorrect assumptions",
        }
        return hints
    
    def predict_with_confidence(self, text):
        """Make prediction with confidence scores"""
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding=True
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
        
        return probs.cpu().numpy()[0]
    
    def get_top_predictions(self, text, top_k=3):
        """Get top-k predictions with labels and probabilities"""
        probs = self.predict_with_confidence(text)
        top_indices = np.argsort(probs)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                'label': self.label_encoder.classes_[idx],
                'probability': probs[idx],
                'rank': len(results) + 1
            })
        return results
    
    def get_attention_weights(self, text):
        """Extract attention weights from the model"""
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding=True
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(
                **inputs,
                output_attentions=True
            )
        
        # Get attention from last layer (averaged over heads)
        attentions = outputs.attentions[-1]
        
        # Average over heads and take CLS token attention to all tokens
        avg_attention = attentions.mean(dim=1)[0]
        cls_attention = avg_attention[0]
        
        # Get tokens
        tokens = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        
        return tokens, cls_attention.cpu().numpy()
    
    def highlight_important_tokens(self, text, top_k=10):
        """Identify and highlight the most important tokens for prediction"""
        tokens, attention_weights = self.get_attention_weights(text)
        
        # Get top-k tokens by attention (excluding special tokens)
        token_importance = []
        for i, (token, weight) in enumerate(zip(tokens, attention_weights)):
            if token not in ['[CLS]', '[SEP]', '[PAD]', '<s>', '</s>', '<pad>']:
                # Clean up subword tokens
                clean_token = token.replace('▁', ' ').replace('Ġ', ' ').strip()
                if clean_token:
                    token_importance.append({
                        'token': clean_token,
                        'weight': float(weight),
                        'position': i
                    })
        
        # Sort by importance
        token_importance.sort(key=lambda x: x['weight'], reverse=True)
        
        return token_importance[:top_k]
    
    def generate_human_explanation(self, text, verbose=True):
        """
        Generate a comprehensive human-readable explanation for the prediction.
        This is the main method for explaining the model's logic to users.
        """
        explanation = {
            'input_text': text,
            'predictions': [],
            'key_evidence': [],
            'reasoning': "",
            'confidence_analysis': "",
            'improvement_suggestions': []
        }
        
        # Get predictions
        top_preds = self.get_top_predictions(text, top_k=3)
        explanation['predictions'] = top_preds
        
        # Get important tokens
        important_tokens = self.highlight_important_tokens(text, top_k=8)
        explanation['key_evidence'] = important_tokens
        
        # Analyze the top prediction
        top_pred = top_preds[0]
        pred_label = top_pred['label']
        confidence = top_pred['probability']
        
        # Parse category and misconception
        if " | " in pred_label:
            category, misconception = pred_label.split(" | ", 1)
        else:
            category = "Unknown"
            misconception = pred_label
        
        # Build reasoning text
        reasoning_parts = []
        
        # 1. State the prediction
        reasoning_parts.append(f"PREDICTED MISCONCEPTION: {misconception}")
        reasoning_parts.append(f"CATEGORY: {category}")
        reasoning_parts.append(f"CONFIDENCE: {confidence*100:.1f}%")
        reasoning_parts.append("")
        
        # 2. Explain key evidence
        reasoning_parts.append("KEY EVIDENCE FROM STUDENT RESPONSE:")
        top_evidence = sorted(important_tokens, key=lambda x: x['weight'], reverse=True)[:5]
        for i, ev in enumerate(top_evidence, 1):
            reasoning_parts.append(f"   {i}. '{ev['token']}' (importance: {ev['weight']:.3f})")
        reasoning_parts.append("")
        
        # 3. Provide conceptual explanation
        reasoning_parts.append("WHY THIS CLASSIFICATION:")
        
        # Check for hints based on keyword matching
        misconception_lower = misconception.lower()
        matched_hints = []
        for keyword, hint in self.misconception_hints.items():
            if keyword in misconception_lower:
                matched_hints.append(hint)
        
        if matched_hints:
            for hint in matched_hints:
                reasoning_parts.append(f"   - {hint}")
        else:
            reasoning_parts.append(f"   - The student's explanation indicates a {category.lower()}-related misunderstanding")
            reasoning_parts.append(f"   - Specifically: {misconception}")
        
        reasoning_parts.append("")
        
        # 4. Confidence analysis
        reasoning_parts.append("CONFIDENCE ANALYSIS:")
        if confidence > 0.8:
            confidence_msg = "High confidence - the model is quite certain about this misconception"
        elif confidence > 0.5:
            confidence_msg = "Moderate confidence - this is the most likely misconception but alternatives are possible"
        else:
            confidence_msg = "Low confidence - consider reviewing alternative misconceptions below"
        reasoning_parts.append(f"   {confidence_msg}")
        
        # Show alternatives if confidence is not very high
        if confidence < 0.9 and len(top_preds) > 1:
            reasoning_parts.append("")
            reasoning_parts.append("ALTERNATIVE POSSIBILITIES:")
            for alt in top_preds[1:]:
                alt_label = alt['label'].split(" | ")[-1] if " | " in alt['label'] else alt['label']
                reasoning_parts.append(f"   - {alt_label} ({alt['probability']*100:.1f}%)")
        
        reasoning_parts.append("")
        
        # 5. Pedagogical suggestions
        reasoning_parts.append("TEACHING RECOMMENDATIONS:")
        reasoning_parts.append(f"   1. Address the specific misconception: {misconception}")
        reasoning_parts.append(f"   2. Review fundamentals related to {category}")
        reasoning_parts.append(f"   3. Provide practice problems targeting this concept")
        
        explanation['reasoning'] = "\n".join(reasoning_parts)
        explanation['confidence_analysis'] = confidence_msg
        
        if verbose:
            print("\n" + "=" * 70)
            print("EXPLAINABLE AI - MISCONCEPTION ANALYSIS")
            print("=" * 70)
            print(explanation['reasoning'])
            print("=" * 70)
        
        return explanation
    
    def visualize_attention(self, text, save_path=None):
        """Create a visualization of attention weights"""
        tokens, attention_weights = self.get_attention_weights(text)
        
        # Filter out padding and special tokens
        valid_indices = [i for i, t in enumerate(tokens) 
                        if t not in ['[PAD]', '<pad>'] and i < len(attention_weights)]
        
        filtered_tokens = [tokens[i] for i in valid_indices]
        filtered_weights = attention_weights[valid_indices]
        
        # Normalize weights for visualization
        filtered_weights = (filtered_weights - filtered_weights.min()) / (filtered_weights.max() - filtered_weights.min() + 1e-8)
        
        # Create figure
        fig, ax = plt.subplots(figsize=(max(12, len(filtered_tokens) * 0.3), 4))
        
        # Create heatmap
        im = ax.imshow(filtered_weights.reshape(1, -1), cmap='YlOrRd', aspect='auto')
        
        # Configure axes
        ax.set_yticks([])
        ax.set_xticks(range(len(filtered_tokens)))
        ax.set_xticklabels([t.replace('▁', '').replace('Ġ', '') for t in filtered_tokens], 
                          rotation=45, ha='right', fontsize=8)
        ax.set_title('Attention Weights - Token Importance for Classification')
        
        plt.colorbar(im, ax=ax, label='Attention Weight')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Attention visualization saved to: {save_path}")
        
        return fig
    
    def explain_single_prediction(self, question, student_answer, student_explanation):
        """
        Convenience method to explain a single prediction with separate inputs.
        This matches the input format of the training data.
        """
        # Format text as expected by the model
        text = f"Question: {question}\nStudent Answer: {student_answer}\nStudent Explanation: {student_explanation}"
        
        return self.generate_human_explanation(text)

In [ ]:
def create_explainer(trainer, tokenizer, label_encoder, config):
    """Factory function to create an explainer from trained model"""
    
    print("\n[Creating Explainable AI Module]")
    print("-" * 40)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    explainer = MisconceptionExplainer(
        model=trainer.model,
        tokenizer=tokenizer,
        label_encoder=label_encoder,
        device=device
    )
    
    print("✓ Explainer created successfully")
    print(f"✓ Device: {device}")
    print(f"✓ Available methods:")
    print("   - generate_human_explanation(text)")
    print("   - explain_single_prediction(question, answer, explanation)")
    print("   - highlight_important_tokens(text)")
    print("   - visualize_attention(text)")
    if LIME_AVAILABLE:
        print("   - explain_with_lime(text)")
    if SHAP_AVAILABLE:
        print("   - explain_with_shap(text)")
    
    return explainer


def demo_explainer(explainer, X_val, num_samples=3):
    """Run a demonstration of the explainer on validation samples"""
    
    print("\n" + "=" * 70)
    print("EXPLAINABLE AI DEMONSTRATION")
    print("=" * 70)
    
    # Get sample texts
    sample_df = X_val.head(num_samples)
    
    for idx, row in sample_df.iterrows():
        text = create_input_text(row)
        true_label = row.get('label', 'Unknown')
        
        print(f"\n{'-'*70}")
        print(f"TRUE LABEL: {true_label}")
        print(f"{'-'*70}")
        
        # Generate explanation
        explanation = explainer.generate_human_explanation(text, verbose=True)
        
        # Check if prediction matches
        pred_label = explanation['predictions'][0]['label']
        is_correct = pred_label == true_label
        
        print(f"\n{'✅ CORRECT' if is_correct else '❌ INCORRECT'}")

## Section 8: Main Execution Pipeline

In [ ]:
def main():
    """Main execution function"""
    
    print("\n" + "=" * 60)
    print("DeBERTa Student Misconception Classification")
    print("with Explainable AI")
    print("=" * 60)
    
    # Initialize
    config = Config()
    seed_everything(config.SEED)
    print_gpu_info()
    
    # Clean GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    
    # Load data
    train_df, label_encoder, num_labels = load_and_preprocess_data(config)
    
    # Setup model
    model, tokenizer = setup_model_and_tokenizer(config, num_labels)
    
    # Prepare datasets
    train_dataset, val_dataset, X_train, X_val = prepare_datasets(
        train_df, tokenizer, config
    )
    
    # Train
    trainer = train_model(model, tokenizer, train_dataset, val_dataset, config)
    
    # Evaluate
    results = evaluate_model(trainer, val_dataset, X_val, label_encoder)
    
    # Save validation predictions for analysis
    val_results = save_validation_predictions(results, X_val, label_encoder, config)
    
    # Save artifacts
    save_artifacts(trainer, label_encoder, config)
    
    # Create Explainable AI module
    explainer = create_explainer(trainer, tokenizer, label_encoder, config)
    
    # Run XAI demonstration on validation samples
    demo_explainer(explainer, X_val, num_samples=3)
    
    # Save explainer for later use
    explainer_path = f"{config.OUTPUT_DIR}/explainer_info.pkl"
    with open(explainer_path, 'wb') as f:
        pickle.dump({
            'model_path': f"{config.OUTPUT_DIR}/final_model",
            'label_encoder_path': f"{config.OUTPUT_DIR}/label_encoder.pkl",
            'config': {
                'model_name': config.MODEL_NAME,
                'max_length': config.MAX_LENGTH
            }
        }, f)
    print(f"\n✓ Explainer info saved to: {explainer_path}")
    
    print("\n" + "=" * 60)
    print("✅ PIPELINE COMPLETE WITH EXPLAINABLE AI!")
    print("=" * 60)
    
    return trainer, results, label_encoder, val_results, explainer

## Run Training Pipeline

In [ ]:
# Execute the main pipeline
trainer, results, label_encoder, val_results, explainer = main()

## Interactive Examples: Using the Explainer

Now that training is complete, you can use the explainer to analyze student misconceptions!

In [ ]:
# Example 1: Explain a custom student response
explanation = explainer.explain_single_prediction(
    question="What is the derivative of x^2?",
    student_answer="x",
    student_explanation="I just removed the exponent"
)

print("\n✅ Explanation generated!")
print("Top prediction:", explanation['predictions'][0]['label'])
print("Confidence:", f"{explanation['predictions'][0]['probability']*100:.1f}%")

In [ ]:
# Example 2: Visualize attention for a sample
sample_text = create_input_text(val_results.iloc[0])
fig = explainer.visualize_attention(sample_text)
plt.show()

In [ ]:
# Example 3: Get token importance for any input
important_tokens = explainer.highlight_important_tokens(sample_text, top_k=10)

print("\nMost Important Tokens:")
print("=" * 50)
for i, token_info in enumerate(important_tokens, 1):
    print(f"{i}. '{token_info['token']}' - Importance: {token_info['weight']:.4f}")

## Load Explainer from Checkpoint (Optional)

If you want to load the explainer later without retraining:

In [ ]:
def load_explainer_from_checkpoint(model_dir):
    """
    Load an explainer from a saved checkpoint.
    
    Usage:
        explainer = load_explainer_from_checkpoint("./deberta_output")
    """
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    
    print(f"Loading explainer from: {model_dir}")
    
    # Load label encoder
    with open(f"{model_dir}/label_encoder.pkl", 'rb') as f:
        label_encoder = pickle.load(f)
    
    # Load config
    with open(f"{model_dir}/config.pkl", 'rb') as f:
        config_dict = pickle.load(f)
    
    # Load model and tokenizer
    model_path = f"{model_dir}/final_model"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    
    # Create explainer
    explainer = MisconceptionExplainer(
        model=model,
        tokenizer=tokenizer,
        label_encoder=label_encoder
    )
    
    print("✓ Explainer loaded successfully!")
    return explainer

# Uncomment to load from checkpoint:
# explainer = load_explainer_from_checkpoint("/kaggle/working/deberta_output")

## Summary

This notebook provides:
1. ✅ **Complete DeBERTa training pipeline** for student misconception classification
2. ✅ **Explainable AI module** with attention visualization and human-readable explanations
3. ✅ **Model evaluation** with MAP@3, accuracy metrics
4. ✅ **Save/load functionality** for later use

### Key Methods:
- `explainer.explain_single_prediction(question, answer, explanation)` - Explain any student response
- `explainer.visualize_attention(text)` - Show attention heatmap
- `explainer.highlight_important_tokens(text)` - Get key words that influenced prediction
- `load_explainer_from_checkpoint(path)` - Load saved model for inference

Happy explaining! 🚀